# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema, accessible at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

We use this schema to discover data structure and to load the records.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

# Also ensure pandas and matplotlib are installed
!pip install pandas matplotlib

## 1. Data Loading
Load the dataset metadata and inspect the dataset-level description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

dataset = mlc.Dataset(croissant_url)

# Display high-level metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"License: {md.license}")
print(f"Date Published: {md.datePublished}")

## 2. Data Overview
Display available record sets and their fields, referencing them by their `@id`.

> In Croissant, a **record set** is a structured table of data. Each record set, its fields, and its columns have unique `@id` identifiers that you should use when referencing them for programmatic access.

Below, we list all record sets in the dataset (by `@id`) and list their fields and columns (also by `@id`).

In [ ]:
# List out all record sets and their contained fields/columns, using @id where possible
overview = []
if dataset.record_sets:
    for rs in dataset.record_sets:
        print(f"RecordSet @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print(f"  Fields (by @id):")
            for f in rs.fields:
                print(f"    - {f.id}")
        if hasattr(rs, 'columns') and rs.columns:
            print(f"  Columns (by @id):")
            for c in rs.columns:
                print(f"    - {c.id}")
        print()
        overview.append(rs.id)
else:
    print("No record sets were found in the Croissant schema.")

# Store for later
record_set_ids = overview

> **Tip:** If the output above says "No record sets were found", it's likely that the Croissant resource defines record sets inside distributions or some nested structure instead of top-level—if so, explore the schema or use `dataset.metadata.to_json()` to investigate the data URLs and structures available.

If record sets exist, use their `@id` in the next step.

## 3. Data Extraction
Extract data from a record set into a DataFrame for further exploration. Use the record set `@id` discovered in the previous overview. All entities (record set, field, column) are referenced by their `@id`.

In [ ]:
# Choose a record set @id to load data from (update this as needed based on the overview above).
from pprint import pprint

# Use first available record_set if any, else, examine metadata for alternate data locations.
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    print(f"Loading records from record set @id: {chosen_record_set_id}")
    records = list(dataset.records(record_set=chosen_record_set_id))
    df = pd.DataFrame(records)
    print("Columns (@id):")
    pprint(df.columns.tolist())
    display(df.head())
else:
    print("No record sets found to extract data. Attempting to print metadata structure for diagnosis...")
    pprint(dataset.metadata.to_json())  # For user inspection

If no records were loaded, please check the schema for alternate structures or data distributions. For this example, we proceed with the available DataFrame.

## 4. Exploratory Data Analysis (EDA)
Explore the dataset: filter records, normalize numeric fields, group/categorize, and handle NA values as part of the data processing pipeline.

We select a numeric field (by its `@id`), filter by a threshold, normalize, and group by another field. Adjust the field `@id`s below to match ones listed in the previous Data Extraction output.

In [ ]:
# Specify the numeric and group fields by their @id.
# Update 'numeric_field_id' and 'group_field_id' to match those output from df.columns
numeric_field_id = '<your_numeric_field_@id>'    # e.g., 'cr:log_likelihood' or similar
group_field_id = '<your_group_field_@id>'        # e.g., 'cr:household_gender' or similar

if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization
Visualize the distribution or relationships between fields in the dataset, using their `@id` for clarity.

Below, we create basic visualizations if sufficient numeric data is present.

In [ ]:
# Visualize data: Histogram and boxplot for the numeric field (by @id)
if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(12, 5))
    plt.subplot(1,2,1)
    df[numeric_field_id].hist(bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')

    plt.subplot(1,2,2)
    df.boxplot(column=numeric_field_id, by=group_field_id if group_field_id in df.columns else None, grid=False)
    plt.title(f'{numeric_field_id} by {group_field_id}')

    plt.tight_layout()
    plt.show()
else:
    print(f"Cannot plot: '{numeric_field_id}' is not in the DataFrame or not numeric.")

## 6. Conclusion
This notebook illustrated how to:
- Load FAIR² data via its Croissant schema (`mlcroissant`)
- List and select record sets and fields by `@id`
- Extract and filter records, normalize numeric columns, and group by key attributes
- Visualize numeric distributions with field identifiers

By using explicit entity `@id` references, all steps are robust, reproducible, and schema-aligned for analytic and FAIR data access.